In [ ]:
# Cài đặt các thư viện cần thiết
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["numpy","pandas","pyreadr","pyarrow","rdata","great_tables","statsmodels", "pygam",
            "scikit-learn" ,"matplotlib" , "qiskit-aer" , "qiskit","qiskit-machine-learning","qkm_swap_test"]:
    try:
        install(pkg)
    except Exception as e:
        print(f"Cảnh báo: {pkg} — {e}")

print("Tất cả thư viện đã sẵn sàng.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH     = '/content/drive/MyDrive/...'   # path đến osf_mini.Rda
VAR_META      = '/content/drive/MyDrive/...'   # path đến osf_var_meta.xlsx
DATA_ALL      = '/content/drive/MyDrive/...'   # path đến dtAll.csv
SAVE_PATH     = '/content/drive/MyDrive/...'   # path lưu artifacts

In [ ]:
import pandas as pd
import numpy as np
import pyreadr

result = pyreadr.read_r(DATA_PATH)
dt_raw = result[None]

var_meta = pd.read_excel(VAR_META)

dtAll = pd.read_csv(DATA_ALL)


# ***II . Feature Extraction***

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf
from pygam import LinearGAM, s
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from scipy.stats import multivariate_normal
import warnings
from tqdm import tqdm
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.packages import importr

# ===========================================================================
# featureExtractor
# ===========================================================================

pandas2ri.activate()
mgcv = importr("mgcv")

def _gam_edf(sub: pd.DataFrame, item: str, tid: str) -> float:
    df = sub[[tid, item]].dropna()
    unique_vals = df[item].nunique()
    if unique_vals <= 2:
        return np.nan
    k = min(unique_vals, 10)
    if k <= 2:
        return np.nan
    try:
        r_df = pandas2ri.py2rpy(
            df.rename(columns={tid: "time", item: "var"})
        )
        formula = ro.Formula(f"var ~ s(time, k={k}, bs='tp')")
        gam_fit = mgcv.gam(formula, data=r_df, method="REML")
        edf = float(ro.r["summary"](gam_fit).rx2("edf")[0])
        return edf
    except Exception:
        return np.nan

def feature_extractor(data, items, pid, tid):

    df = data.copy().rename(columns={pid: "ID", tid: "TIDnum"})
    df = df.sort_values(["ID", "TIDnum"]).reset_index(drop=True)

    # --- Univariate features ------------------------------------------------
    def mac(x):
        v = x.dropna().values
        return np.sum(np.abs(np.diff(v))) / (len(v) - 1) if len(v) > 1 else np.nan

    def rmssd(x):
        v = x.dropna().values
        d = np.diff(v)
        return np.sqrt(np.mean(d ** 2)) if len(d) > 0 else np.nan

    def mad(x):
        med = x.median()
        return (x - med).abs().median() * 1.4826

    agg_records = []
    for pid_val, sub in df.groupby("ID"):
        row = {"ID": pid_val}
        for item in items:
            s_ = sub[item]
            row[f"{item}_mean"]   = s_.mean()
            row[f"{item}_median"] = s_.median()
            row[f"{item}_sd"]     = s_.std(ddof=1)
            row[f"{item}_mad"]    = mad(s_)
            row[f"{item}_rmssd"]  = rmssd(s_)
            row[f"{item}_mac"]    = mac(s_)
        agg_records.append(row)

    feat_univar = pd.DataFrame(agg_records)

    # --- Multivariate / time-based features ---------------------------------
    feat_multivar = feat_univar[["ID"]].copy()
    results_gam = {}

    ids = df["ID"].unique()

    for pid_val in tqdm(ids, desc="Participants"):
        sub = df[df["ID"] == pid_val].reset_index(drop=True)

        for item in items:
            col = item
            if sub[col].notna().sum() == 0:
                continue

            # --- Linear trend -----------------------------------------------
            df_lm = sub[["TIDnum", col]].dropna()
            slope = np.polyfit(df_lm["TIDnum"], df_lm[col], 1)[0] if len(df_lm) >= 2 else np.nan
            feat_multivar.loc[feat_multivar["ID"] == pid_val, f"{item}_lin"] = slope

            # --- Prepare GAM data -------------------------------------------
            df_gam = sub[["TIDnum", col]].rename(columns={"TIDnum": "time", col: "var"})
            var_length = df_gam["var"].dropna().nunique()
            k = min(var_length, 10)
            feat_multivar.loc[feat_multivar["ID"] == pid_val, f"{item}_n"] = var_length

            if k <= 2:
                continue

            # --- EDF via rpy2/mgcv ------------------------------------------
            edf = _gam_edf(sub, item, "TIDnum")
            feat_multivar.loc[feat_multivar["ID"] == pid_val, f"{item}_edf"] = edf

            # --- Autocorrelation: lag-1 và lag-2 ----------------------------
            series_vals = df_gam["var"].values
            try:
                ar_vals = acf(series_vals, nlags=14, fft=False, missing="drop")
                ar01 = ar_vals[1]
                ar02 = ar_vals[2]
            except Exception:
                ar01 = ar02 = np.nan

            feat_multivar.loc[feat_multivar["ID"] == pid_val, f"{item}_ar01"] = ar01
            feat_multivar.loc[feat_multivar["ID"] == pid_val, f"{item}_ar02"] = ar02

            # --- EDF AR1-corrected via rpy2/mgcv ----------------------------
            if not np.isnan(ar01) and k > 3:
                try:
                    df_clean = df_gam.dropna()
                    r_df = pandas2ri.py2rpy(df_clean)
                    formula = ro.Formula(f"var ~ s(time, k={k-1}, bs='tp')")
                    gam_ar = mgcv.gamm(formula, data=r_df)
                    edf_ar = float(ro.r["summary"](ro.r["$"](gam_ar, "gam")).rx2("edf")[0])
                except Exception:
                    edf_ar = np.nan
            else:
                edf_ar = np.nan

            feat_multivar.loc[feat_multivar["ID"] == pid_val, f"{item}_edf_ar"] = edf_ar

        # --- Store per-participant results ----------------------------------
        results_gam[pid_val] = {"origDf": sub}

    # --- Merge univariate + multivariate ------------------------------------
    feat_out = (
        feat_univar.merge(feat_multivar, on="ID")
        .sort_values("ID")
        .reset_index(drop=True)
    )
    other_cols = sorted([c for c in feat_out.columns if c != "ID"])
    feat_out = feat_out[["ID"] + other_cols]

    # --- Standardise --------------------------------------------------------
    feature_cols = [c for c in feat_out.columns if c != "ID"]
    scaler = StandardScaler()
    feat_z = feat_out.copy()
    feat_z[feature_cols] = scaler.fit_transform(feat_out[feature_cols])
    feat_z_mat = feat_z.set_index("ID")

    return {
        "features":     feat_out,
        "featuresZ":    feat_z,
        "featuresZMat": feat_z_mat,
        "gam":          results_gam,
    }


# ===========================================================================
# featureImputer
# ===========================================================================

def feature_imputer(extractor_list, random_state=123):

    feat = extractor_list["features"].copy()
    feature_cols = [c for c in feat.columns if c != "ID"]

    imputer = IterativeImputer(max_iter=50, random_state=random_state)
    feat_imp = feat.copy()
    feat_imp[feature_cols] = imputer.fit_transform(feat[feature_cols])

    scaler = StandardScaler()
    feat_imp_z = feat_imp.copy()
    feat_imp_z[feature_cols] = scaler.fit_transform(feat_imp[feature_cols])
    feat_imp_z_mat = feat_imp_z.set_index("ID")

    return {
        **extractor_list,
        "featuresImp":    feat_imp,
        "featuresImpZ":   feat_imp_z,
        "featuresImpZMat": feat_imp_z_mat,
    }


# ===========================================================================
# pMissTot / pColMiss
# ===========================================================================

def p_miss_tot(x: pd.DataFrame) -> float:
    return x.isna().sum().sum() / (x.shape[0] * x.shape[1]) * 100


def p_col_miss(x: pd.Series) -> float:
    return x.isna().sum() / len(x) * 100


# ===========================================================================
# feature_missing
# ===========================================================================

def feature_missing(features: pd.DataFrame, title: str = "Feature-wise Missingness") -> dict:

    miss_overall = p_miss_tot(features)
    miss_per_feature = pd.DataFrame({
        "feature":      features.columns,
        "perc_missing": [p_col_miss(features[c]) for c in features.columns],
    })

    full_title = f"{title}\n(total missing = {miss_overall:.2f}%)"

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(miss_per_feature["perc_missing"].dropna(),
            bins=range(0, 102, 1), color="black", edgecolor="white", linewidth=0.3)
    ax.set_xlabel("Percent Missing")
    ax.set_ylabel("Number of Features")
    ax.set_title(full_title)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()

    return {
        "miss_overall":         miss_overall,
        "miss_per_feature":     miss_per_feature,
        "plt_miss_per_feature": fig,
    }


# ===========================================================================
# pMissFeat
# ===========================================================================

def p_miss_feat(all_feat, contact, nocontact, title):

    pmiss = pd.DataFrame([
        {"set": "All",        "pMissTotal": p_miss_tot(all_feat)},
        {"set": "Contact",    "pMissTotal": p_miss_tot(contact)},
        {"set": "No Contact", "pMissTotal": p_miss_tot(nocontact)},
    ])

    def _per_feature(df, label):
        return pd.DataFrame({
            "feature": df.columns,
            label:     [p_col_miss(df[c]) for c in df.columns],
        })

    pf = (_per_feature(all_feat, "All")
          .merge(_per_feature(contact,   "Contact"),    on="feature")
          .merge(_per_feature(nocontact, "No Contact"), on="feature"))

    plot_df = (pf.melt(id_vars="feature", var_name="set", value_name="pMiss")
                 .dropna()
                 .merge(pmiss, on="set"))
    plot_df["lab"] = plot_df.apply(
        lambda r: f"{r['set']}\n(total = {r['pMissTotal']:.2f}%)", axis=1)

    sets = plot_df["lab"].unique()
    fig, axes = plt.subplots(1, len(sets), figsize=(5 * len(sets), 4), sharey=True)
    if len(sets) == 1:
        axes = [axes]

    for ax, lab in zip(axes, sets):
        vals = plot_df.loc[plot_df["lab"] == lab, "pMiss"]
        ax.hist(vals, bins=range(0, 102, 1), color="black", edgecolor="white", linewidth=0.3)
        ax.set_title(lab)
        ax.set_xlabel("Percent Missing")
        ax.spines[["top", "right"]].set_visible(False)

    axes[0].set_ylabel("Number of Features")
    fig.suptitle(title)
    plt.tight_layout()

    return {
        "pmiss":        pmiss,
        "pMissFeature": pf,
        "pMissPlot":    fig,
    }

In [ ]:
import pandas as pd

# Lọc chỉ giữ các cột có ít nhất 1 giá trị không phải NA
# (tương đương select_if(~ sum(!is.na(.)) > 1))

feat_data = (
    dtAll
    .sort_values(["ID", "TIDnum"])
    .loc[:, lambda df: df.columns[df.notna().sum() > 1]]
    .reset_index(drop=True)
)

# Lấy danh sách biến cần extract (tương đương var_meta$name[var_meta$cluster == 1])
items = var_meta.loc[var_meta["cluster"] == 1, "name"].tolist()

# Trích xuất features — có thể mất vài giờ với dataset lớn!
feat_full = feature_extractor(
    data=feat_data,
    pid="ID",
    tid="TIDnum",
    items=items,
)

# Xoá tài khoản test nội bộ bị bỏ sót trước đó (ID == 19)
feat_full["features"] = (
    feat_full["features"]
    .loc[feat_full["features"]["ID"] != 19]
    .reset_index(drop=True)
)

In [ ]:
feature_selection = ["ar01", "edf", "lin", "mac", "mad", "median"]

# Chọn các cột kết thúc bằng một trong các suffix (tương đương ends_with)
selected_cols = [
    c for c in feat_full["features"].columns
    if any(c.endswith(f"_{suf}") for suf in feature_selection)
]

perc_miss_features = feature_missing(
    feat_full["features"][selected_cols],
    title="Feature-wise Missingness"
)

perc_miss_features["plt_miss_per_feature"].show()

In [ ]:
feat_full_imp = feature_imputer(feat_full)

In [ ]:
scaled_data = feat_full_imp["featuresImpZMat"][
    [c for c in feat_full_imp["featuresImpZMat"].columns
     if any(c.endswith(f"_{suf}") for suf in feature_selection)]
]

In [ ]:
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

# PCA (scale.=FALSE vì data đã được chuẩn hoá sẵn)
res_pca = PCA()
res_pca.fit(scaled_data)

# Explained variance (tương đương summary(res.pca)$importance)
explained_variance_ratio = res_pca.explained_variance_ratio_

res_var_explained = pd.DataFrame({
    "Components":          range(1, len(explained_variance_ratio) + 1),
    "Explained_Variance":  np.cumsum(explained_variance_ratio),
})

In [ ]:
import pandas as pd

# Tạo DataFrame tóm tắt PCA (tương đương variance_df)
variance_df = pd.DataFrame({
    "Component":               range(1, len(res_pca.singular_values_) + 1),
    "Standard Deviation":      res_pca.singular_values_ / np.sqrt(len(scaled_data) - 1),
    "Proportion of Variance":  res_pca.explained_variance_ratio_,
    "Cumulative Proportion":   np.cumsum(res_pca.explained_variance_ratio_),
})

# Hiển thị bảng (tương đương kable + scroll_box)
variance_df.style \
    .format({
        "Standard Deviation":     "{:.2f}",
        "Proportion of Variance": "{:.2f}",
        "Cumulative Proportion":  "{:.2f}",
    }) \
    .set_caption("PCA Component Summary") \
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "14px"), ("font-weight", "bold"), ("font-family", "Cambria")]},
        {"selector": "th",
         "props": [("font-family", "Cambria"), ("background-color", "#f5f5f5")]},
        {"selector": "tr:hover",
         "props": [("background-color", "#f0f0f0")]},
    ]) \
    .hide(axis="index")

In [ ]:
cutoff_percentage = 0.8

pca_cutoff = int(np.argmax(np.cumsum(res_pca.explained_variance_ratio_) >= cutoff_percentage) + 1)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

explained = res_pca.explained_variance_ratio_ * 100  # chuyển sang %
components = np.arange(1, len(explained) + 1)

fig, ax = plt.subplots(figsize=(16, 8))

# Bars — chỉ highlight đến pca_cutoff (tương đương ncp=pca_cutoff)
colors = ["#a6a6a6" if i < pca_cutoff else "#d9d9d9" for i in range(len(explained))]
# ===== Bars =====
bars = ax.bar(
    components,
    explained,
    color=colors,
    edgecolor="none",
    width=0.7
)

# ===== Line + points =====
ax.plot(
    components,
    explained,
    color="#333333",
    linewidth=1.2,
    marker="o",
    markersize=5,
    markerfacecolor="#333333",
    zorder=3
)

# ===== Percentage labels (only first 15 PCs) =====
for i, (bar, val) in enumerate(zip(bars, explained)):
    if i < 15:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.2,
            f"{val:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9,
            rotation=45,
            color="#333333"
        )

# ===== Axis labels =====
ax.set_xlabel(
    "Principal Components",
    fontsize=16,
    fontweight="bold",
    color="#555555"
)

ax.set_ylabel(
    "Percentage of Explained Variance",
    fontsize=16,
    fontweight="bold",
    color="#555555"
)

ax.set_title(
    "Scree Plot of Explained Variance per Principal Component",
    fontsize=18,
    fontweight="bold",
    color="#444444",
    pad=15
)

# ===== X-axis ticks (show every 5 PCs only) =====
step = 5

ax.set_xticks(components[::step])

ax.set_xticklabels(
    [f"Dim.{i}" for i in components[::step]],
    rotation=45,
    ha="right",
    fontsize=10,
    color="#333333"
)

# ===== Y-axis formatting =====
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter("%.1f"))
ax.tick_params(axis="y", labelsize=11, labelcolor="#333333")

# ===== Theme =====
ax.spines[["top", "right"]].set_visible(False)

ax.yaxis.grid(
    True,
    linestyle="--",
    linewidth=0.5,
    color="#cccccc",
    zorder=0
)

ax.set_axisbelow(True)
ax.minorticks_off()

# ===== Layout =====
plt.tight_layout()

# ===== Show =====
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Tạo DataFrame (tương đương res.varExplained)
res_var_explained = pd.DataFrame({
    "Components":         range(1, len(res_pca.explained_variance_ratio_) + 1),
    "Explained_Variance": np.cumsum(res_pca.explained_variance_ratio_),
})

fig, ax = plt.subplots(figsize=(9, 6))

# geom_point() + geom_line()
ax.plot(res_var_explained["Components"],
        res_var_explained["Explained_Variance"],
        color="#333333", linewidth=1.2, marker="o",
        markersize=5, markerfacecolor="#333333", zorder=3)

# geom_vline (tại pca_cutoff)
ax.axvline(x=pca_cutoff, linestyle=(0, (8, 4)), color="#333333", linewidth=1)

# geom_hline (tại cutoff_percentage)
ax.axhline(y=cutoff_percentage, linestyle=(0, (8, 4)), color="#333333", linewidth=1)

# geom_text: nhãn số components
ax.text(pca_cutoff,
        res_var_explained["Explained_Variance"].min() + 0.01,
        f"{pca_cutoff} components",
        ha="left", va="bottom", fontsize=9, color="#333333")

# geom_text: nhãn % cutoff
ax.text(res_var_explained["Components"].min(),
        cutoff_percentage,
        f"{int(cutoff_percentage * 100)}%",
        ha="left", va="bottom", fontsize=9, color="#333333")

# Labels & title
ax.set_xlabel("Number of Components", fontsize=13, fontweight="bold", color="#555555")
ax.set_ylabel("Proportion of Variance Explained", fontsize=13, fontweight="bold", color="#555555")
ax.set_title("Cumulative Proportion of Variance Explained",
             fontsize=15, fontweight="bold", color="#444444", pad=12)

# theme_Publication
ax.spines[["top", "right"]].set_visible(False)
ax.yaxis.grid(True, linestyle="--", linewidth=0.5, color="#cccccc", zorder=0)
ax.set_axisbelow(True)
ax.tick_params(labelsize=11, labelcolor="#333333")

plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# res.pca$rotation → loadings (components x features), cần transpose để được features x components
loadings = pd.DataFrame(
    res_pca.components_.T,          # shape: (n_features, n_components)
    index=scaled_data.columns,
    columns=[f"PC{i+1}" for i in range(res_pca.n_components_)]
)

# res.pca$x → scores (participants x components)
scores = pd.DataFrame(
    res_pca.transform(scaled_data),  # shape: (n_participants, n_components)
    index=scaled_data.index,
    columns=[f"PC{i+1}" for i in range(res_pca.n_components_)]
)

# --- Plot 1: Variables in reduced space (loadings) -------------------------
plot1 = go.Scatter3d(
    x=loadings["PC1"],
    y=loadings["PC2"],
    z=loadings["PC3"],
    mode="markers",
    marker=dict(size=2),
    text=[f"Variable: {v}" for v in loadings.index],
    hovertemplate="%{text}<extra></extra>",
)

# --- Plot 2: Participants in reduced space (scores) ------------------------
plot2 = go.Scatter3d(
    x=scores["PC1"],
    y=scores["PC2"],
    z=scores["PC3"],
    mode="markers",
    marker=dict(size=2),
    text=[f"Participant: {p}" for p in scores.index],
    hovertemplate="%{text}<extra></extra>",
)

# --- Hiển thị side by side (tương đương htmltools::div) --------------------
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=["Variables in reduced space", "Participants in reduced space"],
)

fig.add_trace(plot1, row=1, col=1)
fig.add_trace(plot2, row=1, col=2)

fig.update_layout(
    width=1200,
    height=600,
    showlegend=False,
    scene=dict(
        xaxis_title="PC1",
        yaxis_title="PC2",
        zaxis_title="PC3",
    ),
    scene2=dict(
        xaxis_title="PC1",
        yaxis_title="PC2",
        zaxis_title="PC3",
    ),
)

fig.show()

In [ ]:
# Lấy PC scores đến pca_cutoff components (tương đương res.pca$x[, 1:pca_cutoff])
pca_scores = scores.iloc[:, :pca_cutoff]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# summarize_all(sd) + pivot_longer
sd_df = (
    pca_scores.std()
    .reset_index()
    .rename(columns={"index": "PC", 0: "sd"})
    .sort_values("sd")  # reorder(PC, sd)
)

fig, ax = plt.subplots(figsize=(8, pca_cutoff * 0.4 + 2))

ax.barh(sd_df["PC"], sd_df["sd"], height=0.5, color="#333333")

ax.set_xlabel("Standard Deviation", fontsize=13, fontweight="bold", color="#555555")
ax.set_ylabel("Principal Component", fontsize=13, fontweight="bold", color="#555555")
ax.set_title("Standard Deviation of each Principal Component",
             fontsize=15, fontweight="bold", color="#444444", pad=12)

# theme_Publication
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", linewidth=0.5, color="#cccccc", zorder=0)
ax.set_axisbelow(True)
ax.tick_params(labelsize=10, labelcolor="#333333")

plt.tight_layout()
plt.show()

In [ ]:
import pickle, pandas as pd

with open(SAVE_PATH, 'wb') as f:
    pickle.dump({
        'pca_scores':  pca_scores,      # DataFrame PC scores
        'scaled_data': scaled_data,     # features trước PCA
        'res_pca':     res_pca,        # PCA model (để transform data mới)
        'pca_cutoff':  pca_cutoff,      # số PC giữ lại
    }, f)
print("Đã lưu toàn bộ artifacts.")

# Load lại
with open(SAVE_PATH, 'rb') as f:
    artifacts = pickle.load(f)

pca_scores  = artifacts['pca_scores']
scaled_data = artifacts['scaled_data']
res_pca     = artifacts['res_pca']
pca_cutoff  = artifacts['pca_cutoff']
print(f"pca_scores shape: {pca_scores.shape}")

In [ ]:
import pickle

# result = feature_extractor(feat_data, items, "ID", "TIDnum")
# result_imputed = feature_imputer(result)

save_data = {
    "features":       feat_full["features"],
    "featuresZ":      feat_full["featuresZ"],
    "featuresImp":    feat_full_imp["featuresImp"],
    "featuresImpZ":   feat_full_imp["featuresImpZ"],
}

with open("features_output.pkl", "wb") as f:
    pickle.dump(save_data, f)

feat_full["features"].to_csv("/content/drive/MyDrive/features.csv", index=False)
feat_full_imp["featuresImp"].to_csv("/content/drive/MyDrive/features_imp.csv", index=False)

In [ ]:
import pickle

# ====================================
# FEATURE EXTRACTION
# ====================================

result = feature_extractor(
    feat_data,
    items,
    "ID",
    "TIDnum"
)

# ====================================
# IMPUTATION
# ====================================

result_imputed = feature_imputer(
    result
)

# ====================================
# SAVE DATA
# ====================================

save_data = {
    "features": result["features"],
    "featuresZ": result["featuresZ"],
    "featuresImp": result_imputed["featuresImp"],
    "featuresImpZ": result_imputed["featuresImpZ"]
}

# ====================================
# SAVE PKL
# ====================================

with open(
    "features_output.pkl",
    "wb"
) as f:

    pickle.dump(
        save_data,
        f
    )

print("Saved features_output.pkl")